# Analise do Dataset da Fase 1

Este notebook integra o dataset `heart_disease_uci_fiap(in).csv` ao contexto da Fase 2.

Objetivos desta etapa:
- carregar a base clinica estruturada da Fase 1;
- observar distribuicao dos pacientes;
- resumir variaveis clinicas relevantes;
- discutir vies e representatividade.

## 1. Importacao de bibliotecas

Nesta analise, o `pandas` e suficiente para leitura, agregacao e resumao dos dados.

In [ ]:
from pathlib import Path

import pandas as pd

## 2. Leitura do dataset

O arquivo abaixo corresponde ao dataset clinico reaproveitado da Fase 1.

In [ ]:
# Ajusta o caminho base para o notebook funcionar dentro ou fora da pasta notebooks.
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = BASE_DIR / 'data' / 'heart_disease_uci_fiap(in).csv'

# Carrega o dataset clinico em DataFrame.
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Dimensao e distribuicao da variavel alvo

A variavel `target` indica a presenca ou ausencia de doenca cardiaca no dataset.

In [ ]:
# Exibe o tamanho da base e a distribuicao entre os dois grupos da variavel target.
df.shape, df['target'].value_counts()

## 4. Criacao de colunas auxiliares

Aqui criamos variaveis mais legiveis para apoiar a discussao de vies por sexo e faixa etaria.

In [ ]:
# Traduz a codificacao da coluna sex para labels textuais.
df['sexo_label'] = df['sex'].map({0: 'feminino', 1: 'masculino'})

# Agrupa idades em faixas para facilitar comparacoes e visualizacao.
df['faixa_etaria'] = pd.cut(
    df['age'],
    bins=[0, 39, 49, 59, 69, 120],
    labels=['ate 39', '40-49', '50-59', '60-69', '70+'],
    include_lowest=True
)

df[['age', 'sexo_label', 'faixa_etaria', 'target']].head()

## 5. Estatisticas clinicas

As colunas abaixo ajudam a resumir o perfil do conjunto de pacientes.

In [ ]:
# Resume variaveis clinicas numericas importantes para o contexto cardiologico.
estatisticas = df[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']].agg(['mean', 'median', 'min', 'max']).round(2)
estatisticas

## 6. Analise por sexo

A ideia aqui e observar quantos pacientes existem por grupo e qual a taxa percentual de `target = 1`.

In [ ]:
# Agrupa por sexo para avaliar representatividade e taxa da variavel alvo.
analise_sexo = df.groupby('sexo_label', observed=False)['target'].agg(['count', 'mean'])
analise_sexo['mean'] = (analise_sexo['mean'] * 100).round(2)
analise_sexo.rename(columns={'count': 'qtd_pacientes', 'mean': 'taxa_doenca_percentual'})

## 7. Analise por faixa etaria

Esse recorte ajuda a discutir se a base esta equilibrada entre idades e como a taxa da variavel alvo se comporta.

In [ ]:
# Agrupa por faixa etaria para comparar quantidade de pacientes e taxa de target.
analise_faixa = df.groupby('faixa_etaria', observed=False)['target'].agg(['count', 'mean'])
analise_faixa['mean'] = (analise_faixa['mean'] * 100).round(2)
analise_faixa.rename(columns={'count': 'qtd_pacientes', 'mean': 'taxa_doenca_percentual'})

## 8. Indicadores de risco observados no dataset

Esta celula resume a frequencia de alguns indicadores clinicos relevantes para a discussao da Fase 2.

In [ ]:
# Calcula a proporcao de alguns sinais clinicos importantes no conjunto de dados.
indicadores = {
    'pressao_alta_repouso_maior_ou_igual_140': round((df['trestbps'] >= 140).mean() * 100, 2),
    'colesterol_maior_ou_igual_240': round((df['chol'] >= 240).mean() * 100, 2),
    'angina_induzida_exercicio': round((df['exang'] == 1).mean() * 100, 2),
    'glicemia_jejum_alta': round((df['fbs'] == 1).mean() * 100, 2),
}

pd.Series(indicadores, name='percentual')

## 9. Leitura critica

- O dataset possui distribuicao relativamente equilibrada entre `target = 0` e `target = 1`.
- Existe predominancia de pacientes do sexo masculino, o que pode influenciar interpretacoes e modelos.
- A analise por faixa etaria ajuda a discutir vies e representatividade.
- Esses dados servem como contexto clinico e epidemiologico para o modulo textual da Fase 2.